In [2]:
import os
import numpy as np
import pickle
import glob 

# Define the path to Nikolai's data
data_path = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP_instances_300nodes_0.15density/rmp/nikolai'

# Load all problem files from the directory
problem_files = glob.glob(os.path.join(data_path, '*.pkl'))
print(f"Found {len(problem_files)} problem files")




Found 10000 problem files


In [ ]:
# Initialize a list to store all problems
history = []

# Load each problem file
for file_path in problem_files:
    try:
        with open(file_path, 'rb') as f:
            problem_data = pickle.load(f)
            history.append(problem_data)
        #print(f"Loaded {file_path}")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

print(f"Successfully loaded {len(history)} problems")

In [3]:
history[0]

{'A': array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 1.],
        [0., 0., 0., ..., 1., 0., 0.],
        [0., 0., 0., ..., 1., 1., 0.]]),
 'b': array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1

In [4]:
import torch
import gzip
import os
import pickle
from tqdm import tqdm

def save_problems(root,problem_data,subfolder,max_problems=10000):
    os.makedirs(f'{root}/{subfolder}/raw', exist_ok=True)
    ips = []
    pkg_idx = 0
    for problem in tqdm(problem_data):
        A = problem['A']
        b = problem['b']
        c = problem['c'] # in the solver we minimize instead of maximize
        x = problem['x']
        ips.append((torch.from_numpy(A).to(torch.float), torch.from_numpy(b).to(torch.float), torch.from_numpy(c).to(torch.float),torch.from_numpy(np.array(x)))) 
        if len(ips) >= 1000:
                with gzip.open(f'{root}/{subfolder}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                    pickle.dump(ips, file)
                    pkg_idx += 1
                ips = []

root = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/'
subfolder = 'MWISP_instances_100nodes_0.15density/sub/'

save_problems(root,history,subfolder)

100%|██████████| 20000/20000 [01:33<00:00, 213.39it/s]


In [5]:
import torch
import gzip
import os
import pickle
from tqdm import tqdm
import numpy as np

def save_problems(root,num_problems,subfolder,max_problems=10000):
    os.makedirs(f'{root}/{subfolder}/raw', exist_ok=True)
    ips = []
    pkg_idx = 0
    for problem in tqdm(range(num_problems)):
        m = 30000  # number of rows
        n = 700  # number of columns
        
        # Create a sparse random matrix with entries 0 or 1
        density = 0.01  # 1% non-zero elements
        nnz = int(m * n * density)  # number of non-zero elements
        
# Generate random indices for non-zero elements
        row_indices = torch.randint(0, m, (nnz,)).long()
        col_indices = torch.randint(0, n, (nnz,)).long()
        indices = torch.stack([row_indices, col_indices])  # Stack to create a 2D tensor [2, nnz]

        values = torch.ones(nnz)  # all non-zero values are 1

        
        # Create sparse tensor
        A = torch.sparse_coo_tensor(indices, values, (m, n))
        
        # Create vectors directly in PyTorch
        b = torch.rand(m)
        c = torch.rand(n)  # in the solver we minimize instead of maximize
        x = torch.zeros(n)
        
        # Append tensors directly (no conversion needed)
        ips.append((A, b, c, x))
        
        if len(ips) >= 50:
                with gzip.open(f'{root}/{subfolder}/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                    pickle.dump(ips, file)
                    pkg_idx += 1
                ips = []
root = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/'
subfolder = 'IF'

save_problems(root,500,subfolder)

100%|██████████| 500/500 [15:55<00:00,  1.91s/it]


In [3]:
import glob
import os
from tqdm import tqdm
import pickle

# Path to the MWISP directory
mwisp_path = '/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP'

# Find all pickle files in all subdirectories
pickle_files = []
for root, dirs, files in os.walk(mwisp_path):
    for file in files:
        if file.endswith('.pkl'):
            pickle_files.append(os.path.join(root, file))

print(f"Found {len(pickle_files)} pickle files")

# Load all pickle files
all_data = []
for file_path in tqdm(pickle_files, desc="Loading pickle files"):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
        all_data.extend(data)

print(f"Loaded {len(all_data)} instances in total")

Found 10 pickle files


Loading pickle files: 100%|██████████| 10/10 [00:18<00:00,  1.87s/it]

Loaded 20000 instances in total


In [4]:
all_data[0]

{'A': array([[1., 1., 1., ..., 0., 0., 0.],
        [1., 1., 1., ..., 0., 0., 1.],
        [1., 1., 1., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 1.],
        [0., 0., 0., ..., 1., 1., 0.],
        [0., 0., 0., ..., 1., 0., 1.]]),
 'b': array([1., 1., 1., ..., 1., 1., 1.]),
 'c': array([ 8.,  5.,  3.,  4.,  1.,  5.,  8.,  2.,  9.,  3.,  4.,  2.,  3.,
         7.,  3.,  9.,  6.,  8.,  6.,  5.,  9.,  3.,  9.,  9.,  7.,  1.,
         8.,  1.,  6.,  2.,  1., 10.,  5.,  1.,  2.,  8.,  8., 10.,  6.,
         6.,  6.,  5., 10.,  3.,  4.,  2.,  9.,  4.,  4.,  1.,  2.,  4.,
        10.,  9.,  3.,  2.,  8.,  8.,  8.,  1.,  6.,  4.,  7.,  1.,  9.,
         6., 10.,  8.,  7.,  9., 10.,  5.,  8.,  2.,  2.,  4.,  8.,  8.,
         8.,  5.,  9.,  3.,  1.,  9.,  4.,  9.,  6.,  2.,  3.,  7.,  2.,
         7.,  4.,  3.,  1.,  9., 10.,  1., 10.,  2.]),
 'x': [0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [7]:
import torch
import gzip
import numpy as np

os.makedirs(f'/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP/raw', exist_ok=True)
ips = []
pkg_idx = 0
for problem in tqdm(all_data):
    A = problem['A']
    b = problem['b']
    c = problem['c'] # in the solver we minimize instead of maximize
    x = problem['x']
    ips.append((torch.from_numpy(A).to(torch.float), torch.from_numpy(b).to(torch.float), torch.from_numpy(c).to(torch.float),torch.from_numpy(np.array(x)))) 
    if len(ips) >= 1000:
            with gzip.open(f'/home/jan/projects/CIIRC/column generation/IPM_MPNN/fac6/MWISP/raw/instance_{pkg_idx}.pkl.gz', "wb") as file:
                pickle.dump(ips, file)
                pkg_idx += 1
            ips = []

100%|██████████| 20000/20000 [07:38<00:00, 43.60it/s] 
